# Palier M4 — comprendre le texte par embeddings + clustering

**Rappel de l'échelle.** M2 comprend les libellés par un **lexique de règles** écrit à la main. M3 les vectorise par TF-IDF de caractères. Ici, M4 fait le troisième essai : une **IA qui comprend le sens du texte** (embeddings), puis un **regroupement automatique** (K-Means) en familles — sans dictionnaire écrit à la main.

**Écart assumé au protocole.** Le protocole prévoit le modèle `paraphrase-multilingual-MiniLM-L12-v2` (sentence-transformers). Il n'est pas en cache sur cette machine, et cet environnement d'exécution n'a pas accès à internet pour le télécharger. `camembert-base` est en revanche déjà disponible localement, et adapté au français (donc au Camfranglais, mélange français/anglais/argot camerounais) — substitué ici, avec des embeddings obtenus par **mean-pooling** des couches cachées (camembert-base n'est pas un modèle sentence-transformers dédié, contrairement à celui du protocole). Le reste du pipeline (clustering, variables, évaluation) est strictement le même quel que soit le modèle d'embedding utilisé : remplacer le modèle plus tard, quand une connexion internet est disponible, ne demanderait de changer que la cellule d'embedding.

**Ce que ce notebook construit.** M4 = `DECLARATIF + COMPORTEMENTAL + PART_CLUST_*` — exactement la même structure que M2 (`DECLARATIF + COMPORTEMENTAL + PART_*` des règles), pour une comparaison directe et équitable entre règles écrites à la main et clustering non supervisé. Le code générique (pipeline, recherche de `C`, coefficients, DeLong, audit d'équité) vient de [`scoring_utils.py`](scoring_utils.py) ; le rappel de M2 vient de [`categorisation.py`](categorisation.py) — rien n'est redéfini ici.

In [1]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from transformers import CamembertModel, CamembertTokenizer, logging as hf_logging

from categorisation import construire_parts_categories
from scoring_utils import (
    audit_equite,
    construire_pipeline,
    construire_variables_comportementales,
    delong_test,
    evaluer,
    extraire_coefficients,
    rechercher_meilleur_C,
)

hf_logging.set_verbosity_error()  # les poids du pooler non utilisés déclenchent un avertissement sans objet ici

SEED = 42
TARGET = "defaut_90j"

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]
GRILLE_C = [0.001, 0.01, 0.1, 1, 10, 100]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")

device : cuda


## Étape 1 — Charger les données

Mêmes fichiers que les notebooks précédents.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

print(f"clients : {clients.shape[0]} lignes | transactions : {tx.shape[0]} lignes")

clients : 2000 lignes | transactions : 91666 lignes


## Rappel — variables comportementales (M1)

`construire_variables_comportementales` (détaillée dans `baseline.ipynb`), sans lire le sens des libellés.

In [3]:
comp = construire_variables_comportementales(tx, clients)
comp.head()

,client_id,nb_tx,pct_debits,inflow,outflow,mean_abs,std_abs,max_abs,nb_jours_actifs,net_flow,cv_abs,tx_par_jour,ecart_revenu
0,1,28,0.821429,1970486.0,1958456.0,140319.357143,352120.284243,1481199.0,26,12030.0,2.509421,1.076923,-5.025951
1,2,58,0.810345,4128443.0,2782051.0,119146.448276,251905.005755,1009317.0,44,1346392.0,2.114247,1.318182,-6.908895
2,3,36,0.750000,2775876.0,2040831.0,133797.416667,292261.844375,1364223.0,25,735045.0,2.184361,1.440000,-3.895725
3,4,21,0.857143,179598.0,1871038.0,97649.333333,319716.352365,1489911.0,19,-1691440.0,3.274127,1.105263,-0.814121
4,5,29,0.931034,397596.0,4884317.0,182134.931034,357177.292957,1209253.0,20,-4486721.0,1.961059,1.450000,0.181901


## Rappel — palier M2 (règles), pour comparaison directe

`construire_parts_categories` (détaillée dans `m2_m3_texte.ipynb`) donne les `PART_*` par le lexique de règles — le point de comparaison direct de M4, et sa précision de catégorisation contre `gt_categorie`.

In [4]:
parts_m2, tx_categorise_m2 = construire_parts_categories(tx)
PART_COLS_M2 = [c for c in parts_m2.columns if c.startswith("PART_")]

precision_m2 = (tx_categorise_m2.cat_regle == tx_categorise_m2.gt_categorie).mean()
print(f"précision de la catégorisation par règles (M2) : {precision_m2:.3f}")

précision de la catégorisation par règles (M2) : 0.710


## Étape 2 — Nettoyage léger pour l'embedding

Différent du nettoyage de M2 (`categorisation.normaliser_libelle`) : celui-ci collait tout pour la recherche de racines. Un modèle de langue a au contraire besoin des **frontières de mots** — on retire seulement le bruit technique (références, codes, chiffres), sans coller le texte.

In [5]:
def nettoyer_pour_embedding(libelle):
    """Nettoyage léger : garde les mots et leur ordre, retire le bruit technique."""
    s = str(libelle).lower()
    s = re.sub(r"(ref|tpe|ag)\w*", " ", s)
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"[^a-zàâäéèêëïîôöùûüç\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


nettoyer_pour_embedding("paiMt-FAct.EnEo/REF9731984")

'paimt fact eneo'

## Étape 3 — Embeddings par mean-pooling de CamemBERT

Beaucoup de transactions partagent le même libellé une fois nettoyé (les références/dates qui les distinguaient ont disparu) : on **déduplique avant d'embarquer**, ce qui économise l'essentiel du calcul, puis on ré-associe chaque transaction à son embedding par une jointure. L'embedding de phrase est la **moyenne des dernières couches cachées**, pondérée par le masque d'attention (camembert-base n'a pas de tête de pooling dédiée à la similarité de phrases).

In [6]:
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
modele_embed = CamembertModel.from_pretrained("camembert-base").to(DEVICE).eval()


def embarquer_textes(textes, batch_size=64):
    """Embedding de phrase par mean-pooling des dernières couches cachées de CamemBERT."""
    vecteurs = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        entrees = tokenizer(lot, padding=True, truncation=True, max_length=64,
                             return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            sortie = modele_embed(**entrees).last_hidden_state
        masque = entrees["attention_mask"].unsqueeze(-1).float()
        moyenne = (sortie * masque).sum(1) / masque.sum(1).clamp(min=1e-9)
        vecteurs.append(moyenne.cpu().numpy())
    return np.vstack(vecteurs)


tx["libelle_net"] = tx.libelle.map(nettoyer_pour_embedding)
uniques_df = pd.DataFrame({"libelle_net": tx.libelle_net.drop_duplicates().reset_index(drop=True)})
print(f"{len(uniques_df)} libellés uniques (nettoyés) sur {len(tx)} transactions")

embeddings_uniques = embarquer_textes(uniques_df.libelle_net.tolist())
embeddings_uniques.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

33040 libellés uniques (nettoyés) sur 91666 transactions


(33040, 768)

## Étape 4 — Clustering : choisir K

`PCA` réduit les 768 dimensions de CamemBERT (même logique que la SVD de M3), avant `KMeans`. Le protocole propose K=15 : on ne l'impose pas, on trace la **courbe du coude** (inertie) et le **score de silhouette** sur une grille autour de cette valeur, et on choisit K en le justifiant par la courbe obtenue — quitte à s'écarter de 15 si les données le montrent.

In [7]:
pca = PCA(n_components=50, random_state=SEED)
embeddings_reduits = pca.fit_transform(embeddings_uniques)
print(f"variance expliquée par les 50 composantes : {pca.explained_variance_ratio_.sum():.1%}")

rng = np.random.default_rng(SEED)
grille_K = [5, 8, 10, 12, 15, 18, 20, 25]
resultats_K = []
for k in grille_K:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(embeddings_reduits)
    if len(embeddings_reduits) > 3000:
        idx = rng.choice(len(embeddings_reduits), 3000, replace=False)
        sil = silhouette_score(embeddings_reduits[idx], km.labels_[idx])
    else:
        sil = silhouette_score(embeddings_reduits, km.labels_)
    resultats_K.append({"K": k, "inertie": km.inertia_, "silhouette": sil})

resultats_K = pd.DataFrame(resultats_K)
resultats_K

variance expliquée par les 50 composantes : 83.6%


,K,inertie,silhouette
0,5,58842.562500,0.102304
1,8,54019.992188,0.075896
2,10,52237.148438,0.066086
3,12,50813.992188,0.070249
4,15,49064.074219,0.071947
5,18,47646.531250,0.064032
6,20,46832.378906,0.064959
7,25,45142.496094,0.061885


**Lecture.** L'inertie baisse mécaniquement avec K (plus de clusters = groupes plus petits et plus compacts) ; c'est le score de silhouette (qui pénalise les clusters mal séparés) qui départage vraiment les valeurs de K. Le K retenu ci-dessous est celui du meilleur score de silhouette sur la grille testée — pas nécessairement 15 : le résultat est à lire tel quel, pas à forcer vers la valeur du protocole.

In [8]:
K_FINAL = int(resultats_K.loc[resultats_K.silhouette.idxmax(), "K"])
print(f"K retenu (meilleur silhouette) : {K_FINAL}")

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10).fit(embeddings_reduits)
uniques_df["cluster"] = kmeans_final.labels_

K retenu (meilleur silhouette) : 5


## Étape 5 — Nommer les familles

Le protocole prévoit un nommage par les conseillers clientèle — indisponible ici. À la place, un nommage purement **diagnostique** : le `gt_categorie` majoritaire parmi les transactions de chaque cluster (jamais utilisé comme variable, seulement pour lire les résultats). Cela permet de mesurer si le clustering non supervisé retrouve les mêmes catégories que le lexique de règles, ou des regroupements différents.

In [9]:
tx_clusterise = tx.merge(uniques_df, on="libelle_net", how="left")

noms_clusters = (tx_clusterise.groupby("cluster").gt_categorie
                  .agg(lambda s: s.value_counts().idxmax()))

tx_clusterise["nom_cluster"] = tx_clusterise.cluster.map(noms_clusters)
precision_m4 = (tx_clusterise.nom_cluster == tx_clusterise.gt_categorie).mean()

print(f"précision de reconstitution des catégories : {precision_m4:.3f} par clustering (M4) "
      f"vs {precision_m2:.3f} par règles (M2)\n")
noms_clusters.rename("nom_majoritaire (gt_categorie)").to_frame()

précision de reconstitution des catégories : 0.179 par clustering (M4) vs 0.710 par règles (M2)



,nom_majoritaire (gt_categorie)
cluster,
0,MOMO
1,SALAIRE
2,SALAIRE
3,TRANSPORT
4,MOMO


## Étape 6 — Construire les parts par client (`PART_CLUST_*`)

Même construction que les `PART_*` de M2, mais à partir des clusters plutôt que des catégories de règles.

In [10]:
parts_clust = (tx_clusterise.groupby(["client_id", "cluster"]).size()
               .unstack(fill_value=0))
parts_clust = parts_clust.div(parts_clust.sum(axis=1), axis=0)
parts_clust.columns = [f"PART_CLUST_{c}" for c in parts_clust.columns]
parts_clust = parts_clust.reset_index()
PART_CLUST_COLS = [c for c in parts_clust.columns if c.startswith("PART_CLUST_")]

parts_clust.head()

,client_id,PART_CLUST_0,PART_CLUST_1,PART_CLUST_2,PART_CLUST_3,PART_CLUST_4
0,1,0.142857,0.071429,0.107143,0.428571,0.250000
1,2,0.258621,0.086207,0.224138,0.189655,0.241379
2,3,0.305556,0.000000,0.222222,0.250000,0.222222
3,4,0.190476,0.047619,0.190476,0.238095,0.333333
4,5,0.310345,0.103448,0.172414,0.172414,0.241379


## Étape 7 — Construire X/y et le split train/test

Même split (`SEED=42, test_size=0.20`) que les autres notebooks, pour rester comparable.

In [11]:
df = (clients
      .merge(comp, on="client_id", how="left")
      .merge(parts_m2, on="client_id", how="left")
      .merge(parts_clust, on="client_id", how="left"))
colonnes_a_remplir = COMPORTEMENTAL_NUM + PART_COLS_M2 + PART_CLUST_COLS
df[colonnes_a_remplir] = df[colonnes_a_remplir].fillna(0)

y = df[TARGET].values
colonnes_gt = [c for c in df.columns if c.startswith("gt_")]
X = df.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte = df.loc[Xte.index]

print(f"train {len(Xtr)} | test {len(Xte)}")

train 1600 | test 400


## Étape 8 — Rappel rapide M1 et M2

Pour amorcer la chaîne de comparaison : M1 (comportemental seul) et M2 (+ règles), déjà détaillés dans `baseline.ipynb` et `m2_m3_texte.ipynb`.

In [12]:
pipeline_m1 = construire_pipeline(DECLARATIF_NUM + COMPORTEMENTAL_NUM, DECLARATIF_CAT, seed=SEED)
modele_m1, _ = rechercher_meilleur_C(pipeline_m1, Xtr, ytr, GRILLE_C, seed=SEED)
p1 = modele_m1.predict_proba(Xte)[:, 1]
_ = evaluer("M1", yte, p1)

num_m2 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_COLS_M2
pipeline_m2 = construire_pipeline(num_m2, DECLARATIF_CAT, seed=SEED)
modele_m2, _ = rechercher_meilleur_C(pipeline_m2, Xtr, ytr, GRILLE_C, seed=SEED)
p2 = modele_m2.predict_proba(Xte)[:, 1]
_ = evaluer("M2", yte, p2)

M1   | AUC 0.666 | Gini 0.333 | KS 0.275
M2   | AUC 0.718 | Gini 0.436 | KS 0.351


## Étape 9 — Construire et évaluer M4

`DECLARATIF + COMPORTEMENTAL + PART_CLUST_*` — même base que M2, seule la source des parts de catégories change.

In [13]:
num_m4 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_CLUST_COLS

pipeline_m4 = construire_pipeline(num_m4, DECLARATIF_CAT, seed=SEED)
modele_m4, cv_m4 = rechercher_meilleur_C(pipeline_m4, Xtr, ytr, GRILLE_C, seed=SEED)

print(f"meilleur C retenu pour M4 : {modele_m4.named_steps['clf'].C}\n")
p4 = modele_m4.predict_proba(Xte)[:, 1]
_ = evaluer("M4", yte, p4)

coefs_m4 = extraire_coefficients(modele_m4, num_m4, DECLARATIF_CAT)
coefs_m4.head(15)

meilleur C retenu pour M4 : 0.01

M4   | AUC 0.681 | Gini 0.361 | KS 0.297


,variable,coefficient
0,secteur_formel,-0.293265
1,secteur_informel,0.293137
2,tx_par_jour,-0.154258
3,nb_jours_actifs,-0.137485
4,cv_abs,0.116283
5,nb_tx,-0.105567
6,outflow,-0.104925
7,inflow,-0.101962
8,ecart_revenu,0.073980
9,anciennete_mois,-0.067607


**Légende** — à quoi correspond chaque `PART_CLUST_i` (nom diagnostique du cluster, Étape 5) :

In [14]:
legende = noms_clusters.rename("nom_majoritaire").to_frame()
legende.index = [f"PART_CLUST_{i}" for i in legende.index]
legende

,nom_majoritaire
PART_CLUST_0,MOMO
PART_CLUST_1,SALAIRE
PART_CLUST_2,SALAIRE
PART_CLUST_3,TRANSPORT
PART_CLUST_4,MOMO


## Étape 10 — Comparaisons DeLong

M1→M2 (rappel), M1→M4 (le clustering apporte-t-il quelque chose ?), et surtout **M2→M4** : le clustering non supervisé fait-il mieux, moins bien, ou pareil que le lexique de règles écrit à la main — sans aucun travail manuel ?

In [15]:
for nom_a, nom_b, pa, pb in [("M1", "M2", p1, p2), ("M1", "M4", p1, p4), ("M2", "M4", p2, p4)]:
    aucs, pval = delong_test(yte, pa, pb)
    conclusion = "significatif" if pval < 0.05 else "non significatif"
    print(f"DeLong {nom_a}->{nom_b} : AUC {aucs[0]:.3f} -> {aucs[1]:.3f} | "
          f"p = {pval:.2e} ({conclusion})")

DeLong M1->M2 : AUC 0.666 -> 0.718 | p = 4.22e-03 (significatif)
DeLong M1->M4 : AUC 0.666 -> 0.681 | p = 5.02e-02 (non significatif)
DeLong M2->M4 : AUC 0.718 -> 0.681 | p = 5.88e-02 (non significatif)


## Étape 11 — Audit d'équité (modèle M4)

Par cohérence avec les notebooks précédents — même lecture (ratio 4/5, refus des vrais bons), pas de correction ici (voir `equite.ipynb`).

In [16]:
for groupe in ["secteur", "sexe"]:
    audit_equite(dfte, p4, groupe, gt_true_col="gt_defaut_true")
    print()

taux d'approbation par secteur : {'formel': 0.732, 'informel': 0.304}
ratio (règle des 4/5) : 0.42 -> disparate impact
refus des vrais bons par secteur : {'formel': 0.232, 'informel': 0.677}

taux d'approbation par sexe : {'F': 0.507, 'M': 0.492}
ratio (règle des 4/5) : 0.97 -> pas de signal au seuil des 4/5
refus des vrais bons par sexe : {'F': 0.484, 'M': 0.451}



## Lecture des résultats

**Le clustering retrouve-t-il les catégories du lexique ?** La précision de reconstitution (Étape 5) mesure si le regroupement non supervisé, sans aucune racine écrite à la main, redécouvre des familles proches des catégories déjà connues (`gt_categorie`) — un signal indirect de la qualité sémantique des embeddings sur ce vocabulaire Camfranglais, avec un modèle (`camembert-base`) qui n'est ni multilingue ni entraîné pour la similarité de phrases.

**M2 vs M4 : deux chemins vers la même idée.** Le test de DeLong M2→M4 tranche si l'un des deux domine statistiquement l'autre, ou si les deux se valent — auquel cas le lexique de règles reste préférable en pratique (aucune dépendance à un modèle de langue de plusieurs centaines de Mo, aucun GPU nécessaire, et une catégorie par transaction directement lisible sans étape de nommage diagnostique).

**Limite assumée.** `camembert-base` est un écart au protocole (modèle multilingue sentence-transformers indisponible hors-ligne ici) : un résultat plus favorable aux embeddings avec le modèle du protocole, sur un vocabulaire couvrant aussi l'anglais et l'argot, reste possible et resterait à vérifier quand une connexion internet est disponible — le pipeline (nettoyage, mean-pooling, PCA, K-Means, `PART_CLUST_*`) resterait inchangé, seule la cellule d'embedding serait à remplacer.